<h1 style="text-align: center;">Classification de données textuelles </h1>


Lors de l’étape d’ingénierie de données textuelles, nous avons vu que diverses opérations pouvaient être appliquées sur les textes et qu’au final il était possible d’obtenir des textes simplifiés. Nous allons, à présent, étudier comment faire de la classification à partir de données textuelles et comment convertir les textes en vecteurs afin de pouvoir classifier des documents textuels.

Ce notebook est en deux parties :  
1. La première explique comment transformer un texte en vecteur pour appliquer un classifieur.  
2. La seconde s’intéresse à la classification à proprement parler.

> **Attention :** La seconde partie est volontairement courte. En effet, à partir du moment où nous disposons d’une matrice de variables prédictives, le principe est le même que pour une classification "classique". Il est donc impératif d’avoir lu au préalable les notebooks sur **Premières classifications** et **Évaluation de modèles**.

> **Remarque importante :** Dans ce notebook, nous utilisons une représentation par **sac de mots** (*Bag of Words*). Ce type de représentation est assez ancien, mais de nouvelles techniques plus performantes comme les **embeddings** sont actuellement utilisées, notamment dans le cadre d’approches d’apprentissage profond. Malgré cela, il est **important de les connaître**, car les principes sous-jacents existent toujours. De plus, il arrive que des approches basées sur l’apprentissage profond se révèlent moins performantes que des approches **traditionnelles**, tout en nécessitant une grande puissance de calcul et un coût énergétique significatif.


# Environnement

Ce notebook peut fonctionner sur plusieurs environnement `Jupyter`. Il n'est pas nécessaire d'utiliser un environnement disposant de GPU.  

**Pour les utilisateurs de Colab :**

Pour pouvoir utiliser votre répertoire `Google Drive`, il est nécessaire de fournir une autorisation. Pour cela il suffit de décommenter et d'exécuter la cellule suivante.

In [ ]:
#from google.colab import drive
#drive.mount('/content/gdrive')

Décommenter les lignes et corriger la ligne `my_local_drive` pour mettre le chemin vers un répertoire spécifique de votre répertoire Google Drive.

In [ ]:
#import sys
#my_local_drive='/content/gdrive/My Drive/Colab Notebooks/ML_FDS'
#sys.path.append(my_local_drive)

Il faut décommenter la ligne suivante pour se positionner dans le répertoire associé.

In [ ]:
#%cd $my_local_drive

# Installation

**Préparation de l'environnement :** avant de commencer, il est important de s'assurer que toutes les bibliothèques nécessaires sont installées dans l'environnement. Dans la prochaine cellule, nous importons toutes les bibliothèques requises pour ce notebook. Si, lors de l'exécution, une bibliothèque est manquante, il faut l'installer en utilisant la commande suivante dans une cellule.  

*! pip install nom_librairie*  

> **Attention :** lors de l'installation d'une bibliothèque, il est recommandé de redémarrer le noyau (kernel) du  notebook afin d'éviter d'éventuels conflits.

**Remarque :** toutes les bibliothèques sont importées au début pour faciliter la lecture. 

In [ ]:
# Utiliser cette cellule pour installer les bibliothèques manquantes.
# Taper simplement : !pip install nom_bibliotheque_manquante
# Exécuter la cellule, puis relancer celle des imports pour vérifier que tout est bien installé.
# Répéter si nécessaire jusqu'à ce que toutes les bibliothèques soient installées.

# !pip install ...

# Ne pas oublier de redémarrer le kernel après l'installation.

# A noter que parfois sur des cellules il y a :
# %%capture dans des cellules
# Cette commande magique permet de rediriger std-out et de ne pas surcharger l'affichage

In [ ]:
# Importation des différentes librairies, classes et fonctions utiles pour le notebook

# Suppression des warnings de Scikit-Learn liés aux futures versions
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Librairies générales
import pandas as pd  # Manipulation de données sous forme de DataFrame
import re  # Manipulation des chaînes de caractères avec des expressions régulières
import time  # Mesure du temps d'exécution
import numpy as np  # Calcul scientifique et manipulation de tableaux
import pickle  # Sauvegarde et chargement d'objets Python
import sys  # Gestion des fonctions et paramètres système

# Librairies pour l'affichage
import matplotlib.pyplot as plt  # Création de graphiques statiques
import seaborn as sns  # Visualisation avancée basée sur Matplotlib

# Librairies Scikit-Learn pour la vectorisation et les pipelines
from sklearn.feature_extraction.text import CountVectorizer  # Vectorisation en sac de mots
from sklearn.feature_extraction.text import TfidfVectorizer  # Vectorisation TF-IDF
from sklearn.base import BaseEstimator, TransformerMixin  # Création de classes personnalisées (estimator et transformer)
from sklearn.pipeline import Pipeline  # Construction de pipelines de traitement
from sklearn.model_selection import train_test_split  # Séparation des jeux d'apprentissage et de test
from sklearn import metrics  # Calcul de métriques d'évaluation
from sklearn.model_selection import cross_val_score  # Validation croisée
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score  # Matrice de confusion et rapport de classification
from sklearn.model_selection import KFold, GridSearchCV  # K-fold et recherche de paramètres par grille

# Classifiers utilisés dans le notebook
from sklearn.svm import SVC  # Support Vector Machine
#from sklearn.linear_model import LogisticRegression  # Régression logistique
#from sklearn.tree import DecisionTreeClassifier  # Arbre de décision
#from sklearn.neighbors import KNeighborsClassifier  # K plus proches voisins
#from sklearn.naive_bayes import MultinomialNB  # Naïve Bayes multinomial
from sklearn.ensemble import RandomForestClassifier  # Forêt aléatoire

# Librairie pour l'optimisation des hyperparamètres
import optuna  # Optimisation des hyperparamètres par recherche bayésienne

# Librairies NLTK pour le traitement du langage naturel
import nltk  # Librairie principale de NLP
from nltk.stem import WordNetLemmatizer  # Lemmatisation des termes
from nltk.stem import PorterStemmer  # Racinisation des termes
from nltk.corpus import stopwords  # Liste des stopwords pour différentes langues
from nltk import word_tokenize  # Découpage en tokens

# Téléchargement des ressources nécessaires de NLTK
nltk.download('wordnet')  # Pour la lemmatisation
nltk.download('stopwords')  # Liste des stopwords
nltk.download('punkt_tab')  # Tokenisation de phrases et mots

# Définition des stopwords anglais
stop_words = set(stopwords.words('english'))


La cellule suivante permet d'ajouter des fonctions utiles notamment d'affichage déjà présentées précédement. 

In [ ]:
# fonctions utilities (affichage, confusion, etc.)
from MLUtils import *

# Vectorisation

L’objectif de la vectorisation est de transformer les documents en vecteurs. Il existe deux approches principales :  

1. L’approche **sac de mots** (*Bag of Words*), dans laquelle il n’y a aucun ordre dans les termes utilisés et qui ne tient compte que du nombre d’occurrences des termes.  
2. L’approche basée sur **TF-IDF**, qui ne tient pas non plus compte de l’ordre des termes, mais qui pondère les valeurs grâce à la mesure **TF-IDF** au lieu de la simple fréquence des termes.

> **Remarque :** L’une des principales limites de ces approches est qu’elles ne prennent pas en compte l’ordre des mots dans un texte, d’où l’appellation *sac de mots*. Bien que les **n-grammes** puissent partiellement atténuer ce problème, il est indispensable d’utiliser d’autres techniques si l’ordre des mots est un élément crucial de l’analyse.


## L'approche Sac de Mots (**Bag of Words**)

La manière la plus simple de représenter un document sous forme de vecteur (*vectorisation*) est d’utiliser les **sacs de mots** (*Bag of Words*). Il existe généralement deux approches pour gérer ces sacs de mots :  

1. **Comptage** : Compter le nombre d’apparitions d’un mot du vocabulaire dans le document.  
2. **TF-IDF** : Prendre en compte la fréquence d’un mot dans le document, tout en pondérant cette fréquence selon le nombre d’apparitions du mot dans l’ensemble des documents.

La première approche utilise la classe `CountVectorizer`, tandis que la seconde utilise la classe `TfidfVectorizer`.

> **Remarque :** `CountVectorizer` et `TfidfVectorizer` génèrent tous deux des matrices creuses (i.e. contenant beaucoup de zéros) lorsque le vocabulaire est large et que les documents contiennent relativement peu de mots en commun. Toutefois, `TfidfVectorizer` pondère les termes en fonction de leur importance relative dans le corpus, ce qui peut améliorer l’efficacité de certains algorithmes de classification en réduisant l’impact des termes trop fréquents.

### la classe CountVectorizer

L’intérêt de la classe `CountVectorizer` est de compter le nombre d’apparitions de chaque mot d’un *vocabulaire* dans un document.

Cette opération se fait en trois étapes principales :  
1. **Création d’une instance** de la classe `CountVectorizer`.  
2. **Appel de la fonction `fit()`** pour apprendre le vocabulaire.  
3. **Appel de la fonction `transform()`** sur un ou plusieurs documents pour les encoder en vecteurs.

> **Remarque :** Il est également possible d’utiliser la fonction `fit_transform()` qui combine les deux opérations en une seule.

La classe `CountVectorizer` permet également d’effectuer un ensemble de pré-traitements sur un document :  
mise en minuscules, suppression des mots inutiles (*stop words*), suppression des signes de ponctuation, etc. Toutefois, elle ne prend pas en charge la lemmatisation ni la recherche des racines des termes (*stemming*).

Principaux paramètres utiles :

- **`lowercase`** (*booléen*) : Convertit le document en minuscules (*défaut=True*).  
- **`token_pattern`** : Élimine les mots trop petits selon un motif donné (*défaut=None*).  
- **`stop_words`** : Permet d’éliminer les *stop words* du document (*défaut=None*).  
- **`analyzer`** : Précise si l’analyse se fait sur les mots, les caractères ou via une fonction personnalisée de pré-traitement (*défaut=’word’*).  
- **`ngram_range`** : Définit la plage des *n-grammes* à utiliser. La valeur par défaut est *(1, 1)*, ce qui signifie qu’un seul mot est pris en compte.  
- **`max_df`** : Ignore les termes dont la fréquence de document est strictement supérieure à un seuil donné (termes trop fréquents) (*défaut=1.0*).  
- **`min_df`** : Ignore les termes dont la fréquence de document est strictement inférieure à un seuil donné (termes peu fréquents) (*défaut=1*).  
- **`max_features`** : Limite le nombre de caractéristiques (*features*) dans le vecteur (*défaut=None*).

La description complète de la fonction `CountVectorizer` est disponible [ici](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

Nous décrirons par la suite comment utiliser ces différents paramètres.

In [ ]:
# Premier exemple sans paramètre
texte = ["This is a simple EXAMPLE ! of CountVectorizer for creating a vector"]

print("Document initial :", texte, '\n')

# Par défaut, conversion en minuscules
vectorizer = CountVectorizer()

# Création du vocabulaire
vectorizer.fit(texte)

# Encodage du document
vector = vectorizer.transform(texte)

# Liste des différents features
print("Les différents features sont :",
      vectorizer.get_feature_names_out(),
      "\n... à noter, tout est converti en minuscules\n")

# Contenu du vocabulaire
print("Vocabulaire :")
print(vectorizer.vocabulary_)

# Affichage de la taille du vecteur de sortie
print("\nTaille du vecteur :", vector.shape, '\n')

# Exemple avec lowercase=False
print("Conversion en mettant lowercase=False")
vectorizer = CountVectorizer(lowercase=False)

# Création du vocabulaire
vectorizer.fit(texte)

# Liste des différents features
print("Les différents features sont :",
      vectorizer.get_feature_names_out(),
      "\n... à noter, les majuscules sont conservées\n")

Il est possible de combiner `fit` et `transform` en utilisant la méthode `fit_transform`, comme le montre l'exemple suivant où nous créons également un DataFrame pour afficher le vecteur résultat.

In [ ]:
texte = ["This is an example, ! of CountVectorizer for creating a vector",
         "This is another example of CountVectorizer",
         "with or without parameters"]

vectorizer = CountVectorizer()
# fit et transform en une opération
X = vectorizer.fit_transform(texte)

# création du dataframe pour affichage
df = pd.DataFrame(
    data=X.toarray(),  # Utilisation directe de X
    columns=vectorizer.get_feature_names_out()  # Utilisation de get_feature_names_out()
)

display(df)

**`token_pattern`**  

Le paramètre *`token_pattern`* peut être utilisé pour filtrer uniquement les mots d’une certaine taille. Il est, par exemple, très utile pour supprimer les termes composés d’un seul caractère. Pour cela, il suffit de spécifier une expression régulière adaptée.

In [ ]:
texte = ["This is an example, ! of CountVectorizer for creating a vector",
         "This is another example of CountVectorizer",
         "with or without parameters"]

print("Le vocabulaire ne contient que des mots qui ont plus de trois caractères :")
vectorizer = CountVectorizer(token_pattern=r'\w{3,}')  # Mots de 3 caractères ou plus

# fit et transform en une opération
X = vectorizer.fit_transform(texte)

# création du dataframe pour affichage
df = pd.DataFrame(
    data=X.toarray(),  # Utilisation directe de X
    columns=vectorizer.get_feature_names_out()  # Utilisation de get_feature_names_out()
)

display(df)

**`stop_words`**  

Le paramètre *`stop_words`* permet de supprimer du vocabulaire les mots appartenant aux **stopwords** d’une langue (e.g. `stop_words='english'`). Il se base sur une liste de stopwords prédéfinie. Il est également possible de spécifier sa propre liste de stopwords.

In [ ]:
texte = ["This is an example, ! of CountVectorizer for creating a vector",
         "This is another example of CountVectorizer",
         "with or without parameters"]

# Exemple avec stopwords en anglais
print("Le vocabulaire ne contient que des mots qui ne sont pas des stopwords anglais :")
vectorizer = CountVectorizer(stop_words='english')

# fit et transform en une opération
X = vectorizer.fit_transform(texte)

# création du dataframe pour affichage
df = pd.DataFrame(
    data=X.toarray(),  # Utilisation directe de X
    columns=vectorizer.get_feature_names_out()  # Utilisation de get_feature_names_out()
)

display(df)

# Exemple avec une liste personnalisée de stopwords
print("Le vocabulaire ne contient que des mots qui ne sont pas dans une liste spécifiée de stopwords (example, vector, creating) :")
vectorizer = CountVectorizer(stop_words=['example', 'vector', 'creating'])

# fit et transform en une opération
X = vectorizer.fit_transform(texte)

# création du dataframe pour affichage
df = pd.DataFrame(
    data=X.toarray(),  # Utilisation directe de X
    columns=vectorizer.get_feature_names_out()
)

display(df)

**`ngram_range`**  

Il est possible de préciser que les features sont composés de **n-grammes** à l’aide du paramètre *`ngram_range`*. Ce dernier spécifie l’intervalle de tailles possibles. Par exemple :  
- *`ngram_range=(1, 2)`* permet d’obtenir des n-grammes de taille 1 et 2 mots.  
- *`ngram_range=(1, 3)`* permet d’obtenir des n-grammes de 1, 2 et 3 mots.  
- *`ngram_range=(3, 3)`* ne génère que des n-grammes de 3 mots.

Par défaut, les n-grammes sont générés à partir des mots. Si l’on souhaite obtenir des n-grammes de caractères, il suffit d’initialiser le paramètre *`analyzer='char'`*.

In [ ]:
texte = ["This is an example of CountVectorizer for creating a vector",
         "This is another example of CountVectorizer",
         "with or without parameters"]

print("n-grammes de mots de taille 1")
vectorizer = CountVectorizer(ngram_range=(1, 1))
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

print("\nn-grammes de mots de taille 1 et 2")
vectorizer = CountVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

print("\nn-grammes de mots de taille 2 et 3")
vectorizer = CountVectorizer(ngram_range=(2, 3))
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

print("\nn-grammes de mots de taille 3")
vectorizer = CountVectorizer(ngram_range=(3, 3))
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

print("\nn-grammes de caractères de taille 1 et 2")
vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 2))
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

**`min_df` et `max_df`**  

- *`min_df`* ignore les termes dont la fréquence de document (présence en % de documents) est strictement inférieure au seuil spécifié.  
  Par exemple, *`min_df=0.55`* exige qu’un terme apparaisse dans au moins **55% des documents** pour être pris en compte dans le vocabulaire.

- *`max_df`* ignore, à l’inverse, les termes dont la fréquence de document est strictement supérieure au seuil spécifié.  
  Ce paramètre est souvent utilisé pour éliminer les termes trop fréquents, comme les mots communs, qui n’apportent pas de valeur discriminante.


In [ ]:
texte = ["This is an example of CountVectorizer for creating a vector",
         "This is another example of CountVectorizer",
         "with or without parameters"]

print("Conserver uniquement les termes apparaissant dans au moins 50% des documents avec min_df=0.5")
vectorizer = CountVectorizer(min_df=0.5)
# fit et transform en une opération
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

print("\nExclure les termes apparaissant dans plus de 50% des documents avec max_df=0.5")
vectorizer = CountVectorizer(max_df=0.5)
# fit et transform en une opération
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

**`max_features`**  

Le paramètre *`max_features`* permet de préciser la taille de sortie du vecteur, c’est-à-dire le **nombre maximal de termes à conserver** dans le vocabulaire.

In [ ]:
texte = ["This is an example of CountVectorizer for creating a vector",
         "This is another example of CountVectorizer",
         "with or without parameters"]

print("Ne conserver que 8 features pour le vocabulaire")
vectorizer = CountVectorizer(max_features=8)
# fit et transform en une opération
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

print("\nPas de contraintes sur la taille du vocabulaire")
vectorizer = CountVectorizer()
# fit et transform en une opération
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

> **Remarque :** L’inconvénient de `CountVectorizer` est qu’il génère des matrices creuses, c’est-à-dire contenant beaucoup de zéros.  
L’exemple suivant illustre le contenu de la matrice précédente, où le **bleu foncé** indique la présence d’une valeur et le **gris clair** représente un zéro.


In [ ]:
texte = ["This is an example of CountVectorizer for creating a vector",
         "This is another example of CountVectorizer",
         "with or without parameters"]

# Initialisation de CountVectorizer
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out())

# Appel de la fonction pour afficher la matrice creuse
afficher_matrice_sparse(X, vectorizer.get_feature_names_out())

### La classe TfidfVectorizer

Le but de l’utilisation de **TF-IDF** (*Term Frequency – Inverse Document Frequency*) est de réduire l’impact des termes apparaissant très fréquemment dans un corpus donné et qui, par conséquent, sont moins informatifs que les autres termes dans le corpus d’apprentissage.  

`CountVectorizer`, en se basant uniquement sur l’occurrence des mots, peut être trop limité. Une alternative consiste à utiliser la mesure **TF-IDF**, dont l’objectif est de pondérer les termes en fonction de leur importance dans un document donné, tout en réduisant l’influence des termes trop fréquents :

$$
tf-idf(d, t) = tf(t) \times idf(d, t)
$$

où $ tf(t) $ représente la **fréquence du terme**, c’est-à-dire le nombre de fois où le terme $t$ apparaît dans le document,  
et $ idf(d, t) $ est la **fréquence inverse du document**, c’est-à-dire le logarithme inverse du nombre de documents contenant le terme $t$.

Pour illustrer **TF-IDF**, considérons un corpus contenant deux documents :

1. **Document 1** : "Le chat dort dans le jardin." (*6 mots*)  
2. **Document 2** : "Le chien joue." (*3 mots*)  

Le nombre de documents est **2**.

**Calcul du TF-IDF pour le terme "le"**  
Le terme **"le"** apparaît dans **les deux documents**, il est donc peu informatif.
1. **TF ("le")** :  
   - Dans **Document 1** :  
     $TF_{\text{doc1}}("le") = \frac{\text{Nombre d’occurrences de "le" dans doc1}}{\text{Nombre total de mots dans doc1}} = \frac{2}{6} = 0.33$  
   - Dans **Document 2** :  
     $TF_{\text{doc2}}("le") = \frac{1}{3} = 0.33$

1. **IDF ("le")** :  
   $IDF("le") = \log\left(\frac{\text{Nombre total de documents}}{\text{Nombre de documents contenant "le"}}\right) = \log\left(\frac{2}{2}\right) = 0$  

1. **TF-IDF ("le")** :  
   - Dans **Document 1** : $TF-IDF_{\text{doc1}}("le") = 0.33 \times 0 = 0$  
   - Dans **Document 2** : $TF-IDF_{\text{doc2}}("le") = 0.33 \times 0 = 0$

**Calcul du TF-IDF pour le terme "chat"**  
Le terme **"chat"** n’apparaît que dans **Document 1**, ce qui le rend plus spécifique et informatif.
1. **TF ("chat")** :  
   - Dans **Document 1** :  
     $TF_{\text{doc1}}("chat") = \frac{1}{6} = 0.17$  
   - Dans **Document 2** :  
     $TF_{\text{doc2}}("chat") = 0$  

1. **IDF ("chat")** :  
   $IDF("chat") = \log\left(\frac{2}{1}\right) \approx 0.69$  

1. **TF-IDF ("chat")** :  
   - Dans **Document 1** : $TF-IDF_{\text{doc1}}("chat") = 0.17 \times 0.69 \approx 0.12$  
   - Dans **Document 2** : $TF-IDF_{\text{doc2}}("chat") = 0 \times 0.69 = 0$

**Calcul du TF-IDF pour le terme "chien"**  
Le terme **"chien"** n’apparaît que dans **Document 2**, ce qui lui donne également un poids plus élevé.
1. **TF ("chien")** :  
   - Dans **Document 1** :  
     $TF_{\text{doc1}}("chien") = 0$  
   - Dans **Document 2** :  
     $TF_{\text{doc2}}("chien") = \frac{1}{3} \approx 0.33$  

1. **IDF ("chien")** :  
   $IDF("chien") = \log\left(\frac{2}{1}\right) \approx 0.69$  

1. **TF-IDF ("chien")** :  
   - Dans **Document 1** : $TF-IDF_{\text{doc1}}("chien") = 0 \times 0.69 = 0$  
   - Dans **Document 2** : $TF-IDF_{\text{doc2}}("chien") = 0.33 \times 0.69 \approx 0.23$

**Résumé des résultats :**

| Terme  | TF-IDF Document 1 | TF-IDF Document 2 |
|--------|-------------------|-------------------|
| **le** | 0                 | 0                 |
| **chat** | 0.12              | 0                 |
| **chien** | 0                 | 0.23              |


Cet exemple illustre que :

1. Les termes communs (**"le"**) ont un poids **nul** dans les deux documents car ils apparaissent partout, ce qui les rend peu utiles pour la classification.
1. Les termes spécifiques (**"chat"** et **"chien"**) ont un **poids élevé** dans leur document respectif car ils apportent une information discriminante.
3. La différence de longueur entre les documents est prise en compte dans le calcul du **TF**, ce qui influence directement le résultat final du **TF-IDF**.


L'utilisation de `TfidfVectorizer` est similaire à celle de `CountVectorizer`. Cette opération se fait en trois étapes :

1. Création d'une instance de la classe `TfidfVectorizer`.  
2. Appel de la fonction `fit()` pour apprendre le vocabulaire.  
3. Appel de la fonction `transform()` sur un ou plusieurs documents afin de les encoder sous forme de vecteurs.

Les paramètres de `TfidfVectorizer` sont assez similaires à ceux de `CountVectorizer`. Une description détaillée est disponible dans la [documentation officielle](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

> **Remarque :** Si `CountVectorizer` a déjà été utilisé, il est possible d'appliquer directement `TfidfTransformer` sur le résultat pour mettre à jour les valeurs en TF-IDF sans réapprendre le vocabulaire.


In [ ]:
texte = ["This is an example of TfidfVectorizer for creating a vector",
         "This is another example of TfidfVectorizer",
         "with or without parameters"]

print("Application de TfidfVectorizer :")
vectorizer = TfidfVectorizer()

# fit et transform en une opération
X = vectorizer.fit_transform(texte)

# création du DataFrame pour affichage
df = pd.DataFrame(
    data=X.toarray(),  # Utilisation directe de X
    columns=vectorizer.get_feature_names_out()  # Utilisation de get_feature_names_out()
)

display(df)

Il est possible d’obtenir l’IDF de chaque terme du vocabulaire via l’attribut `idf_`. Cet attribut contient les valeurs de **IDF** associées à chaque terme extrait par `TfidfVectorizer`.


In [ ]:
print ("Affichage de l'idf de chaque terme du vocabulaire : ");

print(dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_)))

Un exemple combinant différents attributs :

In [ ]:
texte = ["This is an example of TfidfVectorizer for creating a vector",
         "This is another example of TfidfVectorizer",
         "with or without parameters"]

print("Application de TfidfVectorizer avec stop_words='english', "
      "ngram_range=(1, 2), max_df=0.9, min_df=0.1 et max_features=10 :")

# Initialisation de TfidfVectorizer avec plusieurs attributs
vectorizer = TfidfVectorizer(
    stop_words='english',       # Suppression des stopwords en anglais
    ngram_range=(1, 2),         # Extraction des unigrams et bigrams
    max_df=0.9,                 # Ignorer les termes présents dans plus de 90% des documents
    min_df=0.1,                 # Ignorer les termes présents dans moins de 10% des documents
    max_features=10,            # Limiter à 10 le nombre de termes extraits
)

# fit et transform en une seule opération
X = vectorizer.fit_transform(texte)

# Création d’un DataFrame pour afficher la matrice TF-IDF
df_tfidf = pd.DataFrame(
    data=X.toarray(),
    columns=vectorizer.get_feature_names_out()
)


display(df_tfidf)

> **Remarque :** L’un des principaux avantages de `TfidfVectorizer` par rapport à `CountVectorizer` est que les matrices générées contiennent des **valeurs pondérées** (TF-IDF), ce qui rend les termes fréquents moins influents dans la classification.

`CountVectorizer` et `TfidfVectorizer` ne possèdent pas de dictionnaire de stop words en français.

Téléchargez le fichier : **`StopWordsFrench.csv`** et sauvegardez-le dans votre répertoire de travail.  
Cette liste a été obtenue à partir du site : [frenchstopwords](https://referencement-gratuit.and-co.ch/download/liste-stop-words-francais.txt).

In [ ]:
%%capture
!wget https://www.lirmm.fr/~poncelet/Ressources/StopWordsFrench.csv  

La cellule suivante élimine les stopwords en français, extrait des n-grammes d’intervalle **1 à 2**, convertit le texte en **minuscule** et ne retient que **15 features**.

In [ ]:
# Lecture des stopwords français et conversion en liste
list_french_stopwords = pd.read_csv("StopWordsFrench.csv", 
                                    sep=',', 
                                    index_col=0)
list_french_stopwords = list_french_stopwords.values.flatten().tolist()  # Aplatir la liste


texte = ["Au clair de la lune",
         "mon ami Pierrot",
         "Prête-moi ta plume",
         "Pour écrire un mot",
         "Ma chandelle est morte",
         "Je n'ai plus de feu"]

# Initialisation de TfidfVectorizer avec les paramètres 
vectorizer = TfidfVectorizer(lowercase=True,
                             stop_words=list_french_stopwords,
                             ngram_range=(1, 2),
                             max_features=15)

# fit et transform en une seule opération
X = vectorizer.fit_transform(texte)

# Création d’un DataFrame pour afficher la matrice TF-IDF
df = pd.DataFrame(
    data=X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

display(df)

# Prise en compte des prétraitements avant transformation 


Précédemment, nous avons vu qu'il était possible d'appliquer de très nombreux pré-traitements sur les documents. Même si `CountVectorizer` et `TfidfVectorizer` offrent certaines fonctionnalités, celles-ci peuvent s'avérer insuffisantes selon les besoins.

Dans cette section, nous présentons comment les **pipelines** peuvent être utilisés pour mettre en place une chaîne de traitement qui pré-traite les données avant de les convertir en vecteurs.

Considérons la fonction suivante, qui applique un ensemble de pré-traitements sur un document. Par défaut, tous les paramètres sont définis à `False`. Pour activer un pré-traitement spécifique, il suffit de passer son paramètre à `True`.

In [ ]:
def MyCleanText(X, 
                lowercase=False,        # Mettre en minuscule
                removestopwords=False,  # Supprimer les stopwords
                removedigit=False,      # Supprimer les nombres  
                getstemmer=False,       # Conserver la racine des termes
                getlemmatisation=False, # Lemmatization des termes 
                stop_words=None         # Liste de stopwords personnalisée
               ):
    
    # Conversion en chaîne de caractères
    sentence = str(X)
    
    # Suppression des caractères spéciaux et ponctuation
    sentence = re.sub(r'[^\w\s]', ' ', sentence)
    
    # Substitution des espaces multiples par un seul espace
    sentence = re.sub(r'\s+', ' ', sentence, flags=re.I)
    
    # Découpage en mots
    tokens = word_tokenize(sentence)
    
    # Mettre en minuscule
    if lowercase:
        tokens = [token.lower() for token in tokens]
    
    # Suppression des tokens non alphanumériques
    tokens = [word for word in tokens if word.isalnum()]
    
    # Suppression des nombres
    if removedigit:
        tokens = [word for word in tokens if not word.isdigit()]
    
    # Suppression des stopwords
    if removestopwords and stop_words is not None:
        tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    if getlemmatisation:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Stemming
    if getstemmer:
        ps = PorterStemmer()
        tokens = [ps.stem(word) for word in tokens]
    
    # Recomposition de la phrase nettoyée
    sentence = ' '.join(tokens)
  
    return sentence

Les cellules suivantes illustrent différents cas d'utilisation de la fonction `MyCleanText`. La première illustre comment l'utiliser pour un document.

In [ ]:
texte = """This is an example of using the Function MyCleanText before creating a vector created, \
          this text has some problems like 1 c or even numbers like 13 and we have corpora"""

print ("Texte d'origine :\n", texte,'\n')
print ('Utilisation de MyCleanText avec mise en minuscule et suppression des nombres :')
print (MyCleanText(texte,lowercase=True,removedigit=True),'\n')

Un exemple d'utilisation pour plusieurs documents.

In [ ]:
texte = ["This is an example of using the Function MyCleanText before creating a vector created",
         "This example is not illustrate of to deal with several texts",
         "One of the text has some problems with numbers 12 or a character like a",
         "Or maybe not"]

print('Utilisation de MyCleanText avec conversion en minuscule, en prenant les racines, en supprimant les nombres\n')

# Appliquer MyCleanText sur chaque phrase de la liste texte
result = [MyCleanText(t, lowercase=True, getstemmer=True, removedigit=True) for t in texte]

print(result, '\n')

In [ ]:
texte = "it is an example of using the Function MyCleanText before creating a vector"

print ('Utilisation de MyCleanText en supprimant les stopwords\n')
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
print ("Texte d'origine :\n",texte,'\n')
print ("Après suppression des stopwords")
print (MyCleanText(texte,
                   removestopwords=True,
                  stop_words=stop_words),'\n')

**Les estimateurs et Transformers**

Dans Scikit-Learn, un **estimateur** est une classe qui implémente les méthodes `fit`, `predict` et/ou `transform`, permettant d’entraîner un modèle ou de transformer les données. Un estimateur peut être un **modèle d’apprentissage** (comme une régression logistique ou une machine à vecteurs de support) ou un **Transformer**, qui effectue des pré-traitements ou transformations sur les données.

Un **Transformer** est un type particulier d’estimateur qui, après avoir appris certaines règles à l’aide de la méthode `fit`, transforme les données via la méthode `transform`. Par exemple, un Transformer peut :
- **normaliser** ou **mettre à l’échelle** les données,
- **gérer les valeurs manquantes**,
- **réduire la dimensionnalité**, etc.

Scikit-Learn propose de nombreux Transformers pour automatiser ces tâches courantes.

Pour plus d’informations sur les estimateurs et leur utilisation, vous pouvez consulter la documentation officielle :  
[Documentation Scikit-Learn — Développeurs](https://scikit-learn.org/stable/developers/develop.html)

L’interface de base pour un `Transformer` est la suivante :


```python
from sklearn.base import TransformerMixin

class Transfomer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        """
        Apprendre comme transformer les données en fonction des données d'entrées X.
        """
        return self

    def transform(self, X):
        """
        Transformer X dans un nouveau jeu de données Xprime et le retourner.
        """
        return Xprime
``` 

où via la méthode `Transformer.transform` nous pouvons transformer les données initiales.

Nous pouvons ainsi créer notre propre Transformer, `TextNormalizer`, qui applique des pré-traitements sur les données, tels que la suppression des **stopwords**, la récupération des **racines**, etc., en utilisant la fonction `MyCleanText` définie précédemment.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class TextNormalizer(BaseEstimator, TransformerMixin):
    def __init__(self, 
                 removestopwords=False,    # Suppression des stopwords
                 lowercase=False,          # Passage en minuscule
                 removedigit=False,        # Suppression des nombres  
                 getstemmer=False,         # Racinisation des termes 
                 getlemmatisation=False,   # Lemmatisation des termes  
                 stop_words=None           # Liste personnalisée de stopwords
                ):
        
        self.lowercase = lowercase
        self.getstemmer = getstemmer
        self.removestopwords = removestopwords
        self.getlemmatisation = getlemmatisation
        self.removedigit = removedigit
        self.stop_words = stop_words

    def transform(self, X, **transform_params):
        # Nettoyage du texte
        X = X.copy()  # Pour conserver les données d'origine
        return [MyCleanText(text, 
                            lowercase=self.lowercase,
                            getstemmer=self.getstemmer,
                            removestopwords=self.removestopwords,
                            getlemmatisation=self.getlemmatisation,
                            removedigit=self.removedigit,
                            stop_words=self.stop_words) for text in X]

    def fit(self, X, y=None, **fit_params):
        # Rien à apprendre dans le cas d'un simple Transformer
        return self


La cellule suivante illustre l'utilisation de `TextNormalizer`.

In [ ]:
texte = ["This is an example of using TextNormalizer before creating a vector",
        "its convenient to manage all the preprocessing",
        "before applying a vectorization"]

print ("documents originaux\n ",texte,'\n')
# il suffit de créer une instance de la classe TextNormalizer
# d'appliquer fit.transform pour appliquer les pré-traitements
text_normalizer=TextNormalizer(lowercase=True,
                               removestopwords=True,
                               stop_words=stop_words)  
text_cleaned=text_normalizer.fit_transform(texte)
print ("documents après application des pré-traitements (transformation en minuscule, supression des stopwords)")
print (text_cleaned,'\n')    

Il est maintenant possible d'enchaîner des pré-traitement et l'application d'un tf-idf.

In [ ]:
texte = ["This is an example of using TextNormalizer before creating a vector",
         "its convenient to manage all the preprocessing",
         "before applying a vectorization"]

print("Documents originaux :\n", texte, '\n')

# Création d’un objet de la classe TextNormalizer
text_normalizer = TextNormalizer(lowercase=True, removestopwords=True,
                                 stop_words=stop_words)

# Application de fit_transform pour effectuer les pré-traitements
text_cleaned = text_normalizer.fit_transform(texte)

print("Documents après application des pré-traitements (transformation en minuscule, suppression des stopwords) :")
print(text_cleaned, '\n')

# Enchaînement avec un TfidfVectorizer pour extraire des n-grammes de taille 2
tfidf = TfidfVectorizer(ngram_range=(2, 2))
vector_tfidf = tfidf.fit_transform(text_cleaned)

print("Texte transformé en vecteur TF-IDF (bigrammes) :")
print(vector_tfidf.toarray(), '\n')

# Affichage des features extraits
print("Liste des features extraits (bigrammes) :")
print(tfidf.get_feature_names_out(), '\n')

# Transformation du vecteur en matrice pour l’entrée d’un classifieur
print("Transformation du vecteur TF-IDF en matrice :")
print(vector_tfidf.toarray())

La généralisation du principe précédent via un pipeline se fait alors simplement : 

In [ ]:
texte = ["This is an example of using TextNormalizer before creating a vector",
         "its convenient to manage all the preprocessing",
         "before applying a vectorization"]

# Création du pipeline
pipe = Pipeline([("cleaner", TextNormalizer(removestopwords=False,
                                            lowercase=False)),
                 ("vectorizer", TfidfVectorizer(lowercase=False,
                                                ngram_range=(2, 2)))])

# Application du pipeline sur le texte
pipe.fit(texte)
transformed_text = pipe.transform(texte)

# Création du DataFrame pour affichage
df = pd.DataFrame(
    data=transformed_text.toarray(),
    columns=pipe['vectorizer'].get_feature_names_out()  # Remplacement par get_feature_names_out()
)

display(df)

# Classification de documents textuels

Maintenant que nous savons **pré-traiter les données** et construire une **matrice** à partir des textes, nous pouvons passer à l’étape de **classification**.

Le jeu de données que nous allons utiliser est issu de la base de l’[UCI](https://archive.ics.uci.edu/ml/datasets/Sentiment+Labelled+Sentences) et a été créé dans le cadre de l’article "From Group to Individual Labels using Deep Features", Kotzias et al., KDD 2015.

Ce jeu de données contient des phrases d’avis provenant de trois sites différents :  
- **Yelp**,  
- **Amazon**,  
- **IMDb**.  

Pour chacun de ces sites, il y a **500 avis positifs** (valeur = 1) et **500 avis négatifs** (valeur = 0).

Le [site officiel](https://archive.ics.uci.edu/ml/machine-learning-databases/00331/) propose trois fichiers de textes bruts nommés :  
- *amazon_cells_labelled.txt*  
- *imdb_labelled.txt*  
- *yelp_labelled.txt*  

Une version regroupant ces trois fichiers est disponible dans la cellule suivante.

In [ ]:
%%capture
!wget https://www.lirmm.fr/~poncelet/Ressources/ReviewsLabelled.csv

Principales informations sur les données : 

In [ ]:
df = pd.read_csv("ReviewsLabelled.csv", 
                 names=['sentence','sentiment','source'], 
                 header=0,sep='\t', 
                 encoding='utf8')

print ("les 5 premières lignes du fichier :")
display(df[0:5])
print ("la taille du fichier : ", df.shape)
print ("le nombre d'avis différents : \n",
       df['sentiment'].value_counts(),'\n')

df['sentiment'].value_counts().plot(kind='pie', 
                                  figsize=(8,8),
                                  title='Pie Chart', 
                                  fontsize=11, 
                                  legend=True)
plt.show()

print ("Un exemple d'avis \n",df['sentence'][0],'\n')

In [ ]:
# selection des données
X=df.sentence
y=df.sentiment

## **Quel est le meilleur pré-traitement et la meilleure représentation de vecteur ?**

Nous allons maintenant appliquer le pipeline sur les différentes phrases du corpus de données afin de les transformer en une matrice utilisable pour la classification.

In [ ]:
# Création du pipeline
pipe = Pipeline([("cleaner", TextNormalizer()),
                 ("vectorizer", TfidfVectorizer())
                ])

# Application du pipeline sur les données
X_transformed = pipe.fit_transform(X)

# Création du dataframe pour affichage
df_pipe = pd.DataFrame(
    data=X_transformed.toarray(),
    columns=pipe['vectorizer'].get_feature_names_out())  

# Affichage du dataframe
display(df_pipe)

A partir de maintenant tous les choix qui vont être faits vont avoir une influence sur la classification. Pour l'instant, outre les valeurs par défaut de `TfidfVectorizer`(e.g mise en minuscule), aucun prétraitement n'est réalisé. Nous avons vu précédemment qu'il était important d'avoir un premier aperçu de la manière dont les données sont réparties dans l'espace afin de nous guider dans le choix d'un classifier.   

Nous pouvons déjà regarder sans aucun traitement ce que cela donne.

À partir de maintenant, tous les choix réalisés auront une influence sur la classification.  
Pour l'instant, outre les valeurs par défaut de `TfidfVectorizer` (e.g. mise en minuscule), aucun prétraitement supplémentaire n'est appliqué. 

Nous avons vu précédemment qu'il était **important** d'avoir un premier aperçu de la manière dont les données sont réparties dans l'espace, afin de nous guider dans le choix d'un classifier.  

Nous pouvons déjà observer, sans aucun traitement particulier, ce que cela donne.

In [ ]:
# Plot 2D
df_proj_2d = plot_umap(X_transformed, df['sentiment'], n_components=2, filename="umap_2d_1.png")

# Plot 3D
df_proj_3d = plot_umap(X_transformed, df['sentiment'], n_components=3, filename="umap_3d_1.png")

Les données sont très regroupées ... est-ce que d'autres prétraitrements permettraient de mieux les séparer. Nous examinons un peu l'impact des prétraitements sur quelques cas.

In [ ]:
# Création du pipeline
pipe = Pipeline([("cleaner", TextNormalizer(removestopwords=True,
                                                   lowercase=True,
                                                   getstemmer=True,
                                                   removedigit=True)), 
                    ("vectorizer", TfidfVectorizer(lowercase=False))
                ])

# Application du pipeline sur les données
X_transformed = pipe.fit_transform(X)

# Création du dataframe pour affichage
df_pipe = pd.DataFrame(
    data=X_transformed.toarray(),
    columns=pipe['vectorizer'].get_feature_names_out())
# Plot 2D
df_proj_2d = plot_umap(X_transformed, df['sentiment'], n_components=2, filename="umap_2d_2.png")

# Plot 3D
df_proj_3d = plot_umap(X_transformed, df['sentiment'], n_components=3, filename="umap_3d_2.png")

In [ ]:
# Création du pipeline
pipe = Pipeline([("cleaner", TextNormalizer(removestopwords=True,
                                            removedigit=False)), 
                    ("vectorizer", TfidfVectorizer(lowercase=False))
                ])

# Application du pipeline sur les données
X_transformed = pipe.fit_transform(X)

# Création du dataframe pour affichage
df_pipe = pd.DataFrame(
    data=X_transformed.toarray(),
    columns=pipe['vectorizer'].get_feature_names_out())
# Plot 2D
df_proj_2d = plot_umap(X_transformed, df['sentiment'], n_components=2, filename="umap_2d_2.png")

# Plot 3D
df_proj_3d = plot_umap(X_transformed, df['sentiment'], n_components=3, filename="umap_3d_2.png")

Cette dernière étape est **importante**, car elle donne déjà une idée de l'impact des prétraitements. En fonction des prétraitements effectués, nous pouvons constater que certains groupements sont probablement plus faciles à distinguer pour un classifieur.

L’objectif ici n’est pas de rechercher le meilleur prétraitement, mais de montrer comment mettre en œuvre une analyse des données afin d’aboutir, au final, à une classification optimisée. Vous êtes encouragés à expérimenter différents prétraitements pour identifier ceux qui améliorent le mieux les performances du classifieur.

**Un premier essai simple de classification**  

L'objectif ici est de tester un premier classifieur. Nous utiliserons un **SVM** (*Support Vector Machine*), qui donne souvent de bons résultats sur les données textuelles.  

Pour simplifier, nous allons créer un jeu d’apprentissage et un jeu de test, puis évaluer les performances d’un classifieur SVM intégré dans un pipeline utilisant nos derniers prétraitements.

> **Rappel :** L’objectif de cette section n’est pas de trouver le meilleur classifieur, mais de montrer les principes généraux de la classification. Cette partie est volontairement succincte, car nous disposons désormais d’une matrice de variables prédictives (où chaque attribut correspond à un mot, un n-gramme, etc.), et les étapes suivantes reprennent les concepts détaillés dans le notebook **Premières classifications**.


In [ ]:
# Création d'un jeu d'apprentissage et de test
trainsize = 0.7  # 70% pour le jeu d'apprentissage, 30% pour le test
seed = 30
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=trainsize, random_state=seed)

# Création du pipeline en ajoutant le classifieur
pipe = Pipeline([("cleaner", TextNormalizer(removestopwords=True,
                                                   lowercase=True,
                                                   getstemmer=True,
                                                   removedigit=True)), 
                 ("vectorizer", TfidfVectorizer(lowercase=False)),
                 ("SVM", SVC(kernel='linear'))
                ])
# Entraînement du pipeline sur le jeu d'apprentissage
pipe.fit(X_train, y_train)

Prediction pour évaluer la qualité du modèle appris : 

In [ ]:
import seaborn as sns
y_pred = pipe.predict(X_test)
# Affichage du report de classification
print("Report de classification :\n")
print(classification_report(y_test, y_pred))

# Calcul de la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Liste des classes (dans l'ordre des labels utilisés)
classes = np.unique(y_test)

# Affichage de la matrice de confusion avec la fonction personnalisée
plot_confusion_matrix(cm, classes=classes, title='Matrice de confusion')

# Evaluation de différents classifieurs

Dans cette section, nous évaluons différents classifieurs pour voir lequel est le plus performant.  

Comme nous appliquons pour chaque classifier les mêmes pré-traitements (appel de `TextNormalizer` sans paramètres) et l'obtention de la matrice pour tf-idf, plutôt que de faire des pipelines et de relancer cette étape pour chaque classifier nous la réalisons en premier. Puis nous testons les différents classifiers via une cross validation.  

> **Remarque :** le principe est similaire à ce que nous avons vu dans le notebook **Premieres Classification** aussi, pour illustrer, nous nous contentons de ne considérer que deux classifieurs avec un nombre limité de fold dans la cross validation. 

In [ ]:
# creation du tableau des différents classifieurs 
models = []
models.append(('RF', RandomForestClassifier()))
models.append(('SVM', SVC()))

In [ ]:
score = 'accuracy'
seed = 7        
allresults = []
results = []
names = []

X=df.sentence
y=df.sentiment

# Nous appliquons les pré-traitements sur X
text_normalizer=TextNormalizer()  
# Appliquer fit.transform pour réaliser les pré-traitements sur X
X_cleaned=text_normalizer.fit_transform(X)

# Pour l'enchainer avec un tf-idf et obtenir une matrice
tfidf=TfidfVectorizer()
features=tfidf.fit_transform(X_cleaned).toarray()

# Attention ici il faut passer features dans cross_val_score plutôt que X
    
for name,model in models:
    # cross validation en 3 fois
    kfold = KFold(n_splits=3, random_state=seed, shuffle=True)
    
    print ("Evaluation de ",name)
    start_time = time.time()
    # application de la classification
    cv_results = cross_val_score(model, features, y, cv=kfold, scoring=score)
    thetime=time.time() - start_time
    result=Result(name,cv_results.mean(),cv_results.std(),thetime)
    allresults.append(result)
    # pour affichage
    results.append(cv_results)
    names.append(name)
    print("%s : %0.3f (%0.3f) in %0.3f s" % (name, cv_results.mean(), cv_results.std(),thetime))         
    
allresults=sorted(allresults, key=lambda result: result.scoremean, reverse=True) 

# Affichage des résultats
print ('\nLe meilleur resultat : ')
print ('Classifier : ',allresults[0].name, 
       ' %s : %0.3f' %(score,allresults[0].scoremean), 
       ' (%0.3f)'%allresults[0].stdresult,  
       ' en %0.3f '%allresults[0].timespent,' s\n')

print ('Tous les résultats : \n')
for result in allresults:
    print ('Classifier : ',result.name, 
       ' %s : %0.3f' %(score,result.scoremean), 
       ' (%0.3f)'%result.stdresult,  
       ' en %0.3f '%result.timespent,' s')

In [ ]:
plot_comparison_boxplot(results, names)

# Recherche des hyperparamètres et mise en place d'une chaîne de traitement

Nous avons vu que SVM obtenait de bons résultats. Nous allons maintenant rechercher les meilleurs hyperparamètres en utilisant **Optuna**.  

> **Rappel :** L'objectif est d'illustrer le fonctionnement d'une recherche d'hyperparamètres et non pas d'obtenir les meilleurs hyperparamètres possibles. Pour cela, nous limiterons le nombre de valeurs à tester ainsi que le nombre de plis (*k*) pour la validation croisée.

In [ ]:
# Fonction objectif pour Optuna
def objective(trial):
    # Définition des hyperparamètres à tester
    C = trial.suggest_loguniform('C', 1, 10.0)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    gamma = trial.suggest_loguniform('gamma', 7, 10.0) if kernel == 'rbf' else 'scale'
    
    # Création du modèle avec les hyperparamètres suggérés
    model = SVC(C=C, kernel=kernel, gamma=gamma, random_state=seed)
    
    # Cross-validation
    kfold = KFold(n_splits=3, random_state=seed, shuffle=True)
    score = cross_val_score(model, features, y, cv=kfold, scoring='accuracy').mean()
    
    return score

# Création d'une étude Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)  # Limité à 3 essais pour une illustration rapide

# Résultats de la meilleure configuration
print("Meilleurs hyperparamètres :")
print(study.best_params)
print(f"Meilleur score : {study.best_value:.3f}")

In [ ]:
# Fonction objectif pour Optuna
def objective(trial):
    # Suggestions d'hyperparamètres pour TextNormalizer
    getstemmer = trial.suggest_categorical('cleaner__getstemmer', [True, False])
    removedigit = trial.suggest_categorical('cleaner__removedigit', [True, False])
    
    # Suggestions d'hyperparamètres pour TfidfVectorizer
    stop_words = trial.suggest_categorical('tfidf__stop_words', ['english', None])
    lowercase = trial.suggest_categorical('tfidf__lowercase', [True, False])
    
    # Suggestions d'hyperparamètres pour SVC
    C = trial.suggest_categorical('svm__C', [1, 10])
    gamma = trial.suggest_categorical('svm__gamma', [1])
    kernel = trial.suggest_categorical('svm__kernel', ['rbf'])
    
    # Création du pipeline avec les hyperparamètres suggérés
    pipeline = Pipeline([
        ("cleaner", TextNormalizer(getstemmer=getstemmer, removedigit=removedigit)),
        ("tfidf", TfidfVectorizer(stop_words=stop_words, lowercase=lowercase)),
        ('svm', SVC(C=C, gamma=gamma, kernel=kernel))
    ])
    
    # Cross-validation
    kfold = KFold(n_splits=3, random_state=seed, shuffle=True)
    score = cross_val_score(pipeline, X, y, cv=kfold, scoring='accuracy').mean()
    
    return score

# Création de l'étude Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)  # Limité à 3 essais pour illustration

# Résultats de la meilleure configuration
print("Meilleurs hyperparamètres :")
print(study.best_params)
print(f"Meilleur score : {study.best_value:.3f}")

# Sauvegarde du modèle

Dans cette section, nous sauvegardons le modèle pour pouvoir l'utiliser ultérieurement. Pour cela, nous construisons un **pipeline** en utilisant les paramètres et hyperparamètres appris précédemment. 

Nous appliquons ce pipeline, cette fois-ci, à l'ensemble du jeu de données (*X* et *y*) et non plus à un sous-ensemble (*X_train*). Cette étape permet de disposer d'un modèle complet, prêt à être utilisé pour de nouvelles prédictions.


In [ ]:
X=df.sentence
y=df.sentiment
# Récupération des meilleurs hyperparamètres d'Optuna
best_params = study.best_params

# Création du pipeline avec les meilleurs hyperparamètres
pipeline = Pipeline([
    ("cleaner", TextNormalizer(
        getstemmer=best_params['cleaner__getstemmer'],
        removedigit=best_params['cleaner__removedigit']
    )),
    ("vectorizer", TfidfVectorizer(
        stop_words=best_params['tfidf__stop_words'],
        lowercase=best_params['tfidf__lowercase']
    )),
    ("SVM", SVC(
        C=best_params['svm__C'],
        gamma=best_params['svm__gamma'],
        kernel=best_params['svm__kernel']
    ))
])

# Entraînement du pipeline sur l'ensemble des données
pipeline.fit(X, y)

# Sauvegarde du pipeline entraîné dans un fichier
filename='SentimentModel.pkl'
with open(filename, 'wb') as file:
    pickle.dump(pipeline, file)

print("Pipeline sauvegardé sous ", filename)


# Mise en production du modèle : Attention !!

Normalement pour charger le modèle il suffit d'appliquer la cellule suivante : 

In [ ]:
filename='SentimentModel.pkl'
clf_loaded=pickle.load(open(filename,'rb'))

> **Attention :** si vous faites cette opération en dehors du notebook vous devriez avoir l'erreur suivante : 

<IMG SRC="http://www.lirmm.fr/~poncelet/Ressources/missingclass.png" align="center" > 

Actuellement, le modèle utilise une classe (`TextNormalizer`) ainsi qu'une fonction (`MyCleanText`) que vous avez créées pour réaliser les pré-traitements.  
Cependant, lorsque Pickle sauvegarde le modèle, il ne conserve que l’appel à la classe et à la fonction, sans leur corps. En d’autres termes, Pickle ne connaît pas le contenu de la classe ni de la fonction et ne peut donc pas les exécuter lors du chargement du modèle.

Pour permettre au modèle d’accéder à ce contenu, il est nécessaire de créer un fichier, par exemple `CleanText.py`, contenant la définition complète de la classe et de la fonction.

> **Attention :** `TextNormalizer` et `MyCleanText` utilisent également des librairies externes. Ces dernières doivent être correctement importées dans le fichier.

L'importation de la classe et de la fonction se fait en précisant leur nom, comme suit :

```python
from CleanText import TextNormalizer, MyCleanText
```

ou par 
```python
from CleanText import *
```